# 对象类型、类型别名与接口

学习目标：能为对象数据定义可复用契约，区分属性缺失、只读访问与结构检查。

前置知识：JavaScript 对象、属性、方法和递归函数；TypeScript 数组、可选值与类型标注。

适用版本：TypeScript 7.0.2、Node.js 24.11.0；ES 模块，开启 strict。本章附加选项：exactOptionalPropertyTypes=true、noUncheckedIndexedAccess=true，含义见对应知识点。

环境准备：[环境配置与运行](README.md)。

工作目录：content/编程语言/typescript。

配套脚本：位于 scripts/04-objects-and-interfaces/。

1. [main.ts](scripts/04-objects-and-interfaces/main.ts)：按正文顺序组织的正常示例，片段依赖同文件前文定义。
2. [type-errors.ts](scripts/04-objects-and-interfaces/type-errors.ts)：与正常示例隔离的类型反例，不生成或执行 JavaScript。
3. [tsconfig.json](scripts/04-objects-and-interfaces/tsconfig.json)、[tsconfig.errors.json](scripts/04-objects-and-interfaces/tsconfig.errors.json)：分别明确正常与反例文件范围。



Step 1：检查正常项目的类型。

```bash
npm run check:04
```

Step 2：生成正常项目的 JavaScript。

```bash
npm run build:04
```

Step 3：运行正常示例。

```bash
npm run run:04
```

Step 4：检查下文独立列出的类型反例。

```bash
npm run errors:04
# 预期非零退出；按反例注释逐行核对具体错误，不运行 type-errors.ts。
```

正常配置只包含上面列出的正常与独立运行示例，生成文件位于 .build/04-objects-and-interfaces/。错误配置继承正常选项，改用 type-errors.ts 并开启 noEmit。

## 1 对象结构、type 与 interface

对象类型列出需要的成员及其类型。可以直接在参数位置写结构，也可以用 type 起类型别名，或用 interface 声明接口。它们不会创建对象或执行校验。类型别名还可以命名联合等非对象类型；接口侧重对象结构，并支持扩展与声明合并，合并规则在专章展开。

下面的 Named 与 NamedAlias 描述同样的结构；不同名字没有自动建立互不兼容的身份。方法写成 summary(): string，表示接收零个普通参数、返回字符串。

```typescript
export {};
interface Named { title: string; }
type NamedAlias = { title: string };
interface Lesson extends Named {
  minutes: number;
  summary(): string;
}
const lesson: Lesson = {
  title: "接口", minutes: 20,
  summary() { return this.title + ":" + this.minutes; },
};
const named: NamedAlias = lesson;
function titleOf(value: { title: string }): string { return value.title; }
console.log(titleOf(named), lesson.summary()); // 接口 接口:20
```

以下片段来自独立的 type-errors.ts：

```typescript
interface NeedsTitle { title: string; }
const missing: NeedsTitle = {}; // 缺少必需属性 title。
interface WrongExtension extends NeedsTitle { title: number; } // 扩展属性必须兼容原接口。
```

## 2 可选属性与显式 undefined

本章额外开启 exactOptionalPropertyTypes。nickname?: string 表示可省略属性，但属性存在时只能写字符串；nickname: string | undefined 则要求属性存在，值允许 undefined。若两者都允许，要写 nickname?: string | undefined。

读取可选属性仍需考虑 undefined，因为省略属性时读取也得到这个值。属性是否存在与读取结果是否为 undefined 是两个问题；in 运算可以观察这种差异。该选项依赖严格空值检查，不能把它理解为所有可选值都不再含 undefined。

```typescript
interface OptionalName { nickname?: string; }
interface PresentName { nickname: string | undefined; }
interface FlexibleName { nickname?: string | undefined; }
const omitted: OptionalName = {};
const present: PresentName = { nickname: undefined };
const flexible: FlexibleName = { nickname: undefined };
function displayName(value: OptionalName): string {
  return value.nickname === undefined ? "匿名" : value.nickname;
}
console.log(displayName(omitted), "nickname" in omitted, "nickname" in present, "nickname" in flexible); // 匿名 false true true
```

以下片段来自独立的 type-errors.ts：

```typescript
interface OptionalOnly { nickname?: string; }
const explicitUndefined: OptionalOnly = { nickname: undefined }; // 开启 exactOptionalPropertyTypes 后不允许。
const mustExist: { nickname: string | undefined } = {}; // 联合含 undefined 不等于属性可省略。
```

## 3 readonly 属性不递归冻结

readonly 禁止通过该属性重新赋值；若属性指向对象，内部成员能否修改取决于内部类型。它也不会在 JavaScript 中添加访问控制。

在普通对象属性的兼容性比较中，readonly 不会阻止可写别名的出现。因此它适合表达调用方的使用约定，不能充当运行时冻结或安全隔离。只读数组接口还缺少 push 等写入方法，因此不能直接赋给需要这些方法的可写数组类型。

```typescript
interface View {
  readonly id: number;
  readonly detail: { visits: number };
}
const mutable = { id: 1, detail: { visits: 0 } };
const view: View = mutable;
view.detail.visits += 1;
mutable.id = 2;
console.log(view.id, view.detail.visits); // 2 1
```

以下片段来自独立的 type-errors.ts：

```typescript
const locked: { readonly id: number } = { id: 1 };
locked.id = 2; // readonly 阻止此处重新赋值。
```

## 4 索引签名表达字典

未知属性名、已知值类型时，可以写索引签名。[name: string]: number 中 name 仅是键参数名，表示任意字符串键，其属性值必须兼容 number。已命名的字符串属性也受该签名约束。

本例同时开启 noUncheckedIndexedAccess，让未明确声明的字典键读取包含 undefined。字符串索引不保证任意键实际存在。若同时声明 number 与 string 索引，数字索引的值类型必须兼容字符串索引的值类型，因为普通对象的数值属性键会转换为字符串。

```typescript
interface Counts {
  [name: string]: number;
  total: number;
}
const counts: Counts = { total: 3, types: 2 };
interface Labels {
  [index: number]: string;
  [key: string]: string | number;
  length: number;
}
const labels: Labels = { 0: "类型", length: 1 };
console.log(counts.total, counts["missing"] ?? 0, labels[0]); // 3 0 类型
```

以下片段来自独立的 type-errors.ts：

```typescript
interface WrongDictionary {
  [key: string]: number;
  title: string; // 已命名的属性也必须符合字符串索引签名。
}
```

## 5 递归对象结构与遍历

目录树中每个节点的 children 仍是同类节点，可以让接口引用自身；类型别名也能描述递归对象。递归类型不是循环执行，实际遍历需要函数与终止条件。

下面把缺省 children 作为叶节点，仅处理有限、无环的树。这个结构类型本身不排除循环引用；本例输入按约定为有限、无环的树，不额外编写校验入口。

```typescript
type Topic = { title: string; children?: Topic[] };
function countTopics(topic: Topic): number {
  let count = 1;
  for (const child of topic.children ?? []) count += countTopics(child);
  return count;
}
const tree: Topic = { title: "语言", children: [{ title: "类型" }, { title: "函数" }] };
console.log(countTopics(tree)); // 3
```

## 6 额外属性检查的适用位置

额外属性检查（excess property checking）会在新鲜对象字面量直接被赋给目标类型或作为实参时发现多出的属性，有助于捕获拼写错误。它不是所有对象都执行的精确键集合检查。

先存入变量的对象，在成员兼容时可赋给只要求部分成员的类型；其他属性仍在运行时存在。不要通过加变量或断言掩盖真实拼写错误。完全由可选成员组成的目标还有“没有共同属性”的检查，不能概括成变量总能绕过检查。

```typescript
interface Options { title: string; }
const detailed = { title: "对象", minutes: 12 };
const options: Options = detailed;
console.log(options.title, "minutes" in options); // 对象 true
```

以下片段来自独立的 type-errors.ts：

```typescript
interface KnownOptions { title: string; }
const extra: KnownOptions = { title: "对象", minutes: 12 }; // 新鲜字面量多出 minutes。
interface WeakOptions { minutes?: number; }
const unrelated = { duration: 12 };
const weak: WeakOptions = unrelated; // 与全可选目标没有任何共同属性。
```

## 本章小结

type 与 interface 命名的是结构约束，不是运行时转换。可选、只读、索引签名分别约束不同方面；extra property checking 的触发位置也必须结合表达式判断。

## 练习

1. 定义三种配置，分别要求属性可缺失、属性必有但允许 undefined、两者都允许。为每种配置检查空对象和显式 undefined，诊断应与设计一致。
2. 为 Topic 加可选分钟数，递归求和；缺省分钟数计 0，给出两层树并核对总和。
3. 函数要求 { title: string }，分别直接传入和经变量传入 { titel: "课程" }；解释两种位置为什么都不能提供必需的 title，最终修正属性名而不使用断言。

### 提示

1. 保持 exactOptionalPropertyTypes 开启，分别测试 nickname?: string、nickname: string | undefined、nickname?: string | undefined。
2. 以当前节点的 minutes ?? 0 为初值，再累计各子树。
3. 明确被拼错的是必需属性；经变量传入也不会自动补上缺失的必需成员。


### 参考解析

1. 空对象依次通过、失败、通过；显式 nickname: undefined 依次失败、通过、通过。
2. 例如根 5 分钟、两个孩子分别 10 分钟和省略分钟数，总和应为 15；叶节点 children 缺失按空列表处理。
3. 函数要求 title，而传入只有 titel 的对象，两种位置都失败：字面量有额外错误拼写，变量仍缺 title。修正为 title 后都通过；若仅是可选成员拼错，还须结合是否有共同属性判断。


## 参考与引用来源

| 来源 | 支持的知识点与定位 |
| --- | --- |
| TypeScript 官方文档 | [Object Types：属性修饰符、索引签名、扩展与额外属性检查](https://www.typescriptlang.org/docs/handbook/2/objects.html)；[Everyday Types：类型别名与接口](https://www.typescriptlang.org/docs/handbook/2/everyday-types.html#type-aliases)；[exactOptionalPropertyTypes](https://www.typescriptlang.org/tsconfig/exactOptionalPropertyTypes.html)；[noUncheckedIndexedAccess](https://www.typescriptlang.org/tsconfig/noUncheckedIndexedAccess.html)；[3.7：递归类型别名](https://www.typescriptlang.org/docs/handbook/release-notes/typescript-3-7.html#more-recursive-type-aliases)。 |
| npm 官方文档 | [npm run（v11）](https://docs.npmjs.com/cli/v11/commands/npm-run/)：从本技术目录运行已配置脚本，并解析本地工具。 |
